# Red neuronal para predicción de área de incendios forestales
Este notebook muestra los pasos comentados para entrenar y evaluar una red neuronal que predice la variable *area* a partir del dataset **forestfires.csv**.

## 1. Importar librerías
Importamos las librerías necesarias para manipulación de datos, preprocesamiento, modelado y evaluación.

In [35]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RepeatedKFold

## 2. Cargar el dataset
Leemos el archivo CSV con pandas y separamos la variable objetivo `area`.

In [36]:
data = pd.read_csv('forestfires.csv')  # Leer datos
# Separar características (X) y variable objetivo (y)
X = data.drop('area', axis=1)  # Todas menos 'area'
raw_y = data['area']           # Área sin transformar

## 3. Transformar la variable objetivo
Aplicamos `log(1 + area)` para controlar la asimetría de la variable *area*.

In [37]:
y = np.log1p(raw_y)  # Transformación logarítmica

## 4. Normalizar las entradas
Usamos MinMaxScaler para escalar las variables numéricas entre 0 y 1.

In [38]:
# Definir el escalador
scaler = MinMaxScaler()
# Columnas a normalizar
columns_to_normalize = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
X[columns_to_normalize] = scaler.fit_transform(X[columns_to_normalize])

## 5. Codificar variables categóricas
Convertimos `month` y `day` en variables dummy (one-hot encoding).

In [ ]:
# one hot encoding the mes y dia
X = pd.get_dummies(X, columns=['month', 'day'], drop_first=True).astype(float)

# convertir a arrays de numpy
X = X.values
y = y.values

## 6. Definir modelo
- Cross validation con 10 particiones y 30 repeticiones.
- Definimos Red con capas internas en embudo 20-10-5, función logística y optimizador de descenso de gradiente.
- Habilitamos *early stopping* para detener entrenamiento si no mejora la validación.

In [ ]:
# 10-fold cross validation 30 iteraciones
rkf = RepeatedKFold(n_splits=10, n_repeats=30, random_state=42)

mlp = MLPRegressor(
    hidden_layer_sizes=(20, 10, 5),  # Capas ocultas
    activation='logistic',           # Función de activación sigmoidal
    solver='sgd',                    # Descenso de gradiente estocástico
    max_iter=1000,                   # Máximo de iteraciones
    early_stopping=True,             # Detener si no mejora
    random_state=None                # Inicialización aleatoria
)

## 7. Entrenar y evaluar
Iteramos sobre cada partición, entrenamos el modelo, predecimos y calculamos métricas tanto en escala logarítmica como en escala original.

In [28]:
# Listas para métricas en escala logarítmica
mse_list, mae_list, r2_list = [], [], []
# Listas para métricas en escala original
mse_orig, mae_orig, r2_orig = [], [], []
for train_idx, test_idx in rkf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    # Escala logarítmica
    mse_list.append(mean_squared_error(y_test, y_pred))
    mae_list.append(mean_absolute_error(y_test, y_pred))
    r2_list.append(r2_score(y_test, y_pred))
    # Reconversión a escala original
    area_pred = np.expm1(y_pred)
    area_true = raw_y.iloc[test_idx].values
    mse_orig.append(mean_squared_error(area_true, area_pred))
    mae_orig.append(mean_absolute_error(area_true, area_pred))
    r2_orig.append(r2_score(area_true, area_pred))

## 8. Resultados finales
Calculamos los promedios de las métricas para ambas escalas y las mostramos.

In [29]:
# Métricas promedio en escala logarítmica
print('Métricas en escala logarítmica:')
print(f'Average MSE: {np.mean(mse_list):.4f}')
print(f'Average MAE: {np.mean(mae_list):.4f}')
print(f'Average R2: {np.mean(r2_list):.4f}')
# Métricas promedio en escala original
print('Métricas en escala original:')
print(f'Average MSE: {np.mean(mse_orig):.4f}')
print(f'Average MAE: {np.mean(mae_orig):.4f}')
print(f'Average R2: {np.mean(r2_orig):.4f}')

Métricas en escala logarítmica:
Average MSE: 1.9751
Average MAE: 1.1495
Average R2: -0.0304
Métricas en escala original:
Average MSE: 4160.6760
Average MAE: 12.9536
Average R2: -0.0895
